In [1]:
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadMergeEnable\BoSSSpad.dll"
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadPrintRemoveDtRestriction\BoSSSpad.dll"
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadPrintRemoveDtRestrictionAMR\BoSSSpad.dll"
#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadPrintRemoveDtRestrictionAMRDebug\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Text;
using System.Globalization;
using System.Threading;
using System.Threading.Tasks;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using ZwoLevelSetSolver;
using ZwoLevelSetSolver.SolidPhase;
Init();

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
//ExecutionQueues

In [3]:
static var myBatch = ExecutionQueues[2];
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("Droplet-EE-LikeEL-3D-Merge-Compare");

In [4]:
BoSSSshell.WorkflowMgm.Init("Droplet-EE-LikeEL-3D-Merge-Compare");

Project name is set to 'Droplet-EE-LikeEL-3D-Merge-Compare'.
Default Execution queue is chosen for the database.
Opening existing database '\\dc3\userspace\miao\cluster\Droplet-EE-LikeEL-3D-Merge-Compare'.


In [5]:
BoSSSshell.WorkflowMgm.DefaultDatabase = myDb;

## Create grid

In [7]:
public static class GridFactory {
 
    public static Grid3D GenerateGrid() { 
        double xSize = 1;
        double yTop = 1 - 20.0 / 176.7;
        double yBottom = -20.0 / 176.7;
        int kelem = 6 * 4;

        double[] Xnodes = GenericBlas.Linspace(-xSize, xSize, kelem + 1);
        double[] Ynodes = GenericBlas.Linspace(yBottom, yTop, kelem / 2 + 1);
        double[] Znodes = GenericBlas.Linspace(-xSize, xSize, kelem + 1);
                var grd = Grid3D.Cartesian3DGrid(Xnodes, Ynodes, Znodes);

                grd.EdgeTagNames.Add(1, "wall_lower");
                grd.EdgeTagNames.Add(2, "pressure_outlet_upper");
                grd.EdgeTagNames.Add(3, "FreeSlip_left");
                grd.EdgeTagNames.Add(4, "FreeSlip_right");
                grd.EdgeTagNames.Add(5, "FreeSlip_front");
                grd.EdgeTagNames.Add(6, "FreeSlip_back");

                grd.DefineEdgeTags(delegate (double[] X) {
                    byte et = 0;
                    if(Math.Abs(X[1] - yBottom) <= 1.0e-8)
                        et = 1;
                    if(Math.Abs(X[1] - yTop) <= 1.0e-8)
                        et = 2;
                    if(Math.Abs(X[0] + xSize) <= 1.0e-8)
                        et = 3;
                    if(Math.Abs(X[0] - xSize) <= 1.0e-8)
                        et = 4;
                    if(Math.Abs(X[2] - xSize) <= 1.0e-8)
                        et = 5;
                    if(Math.Abs(X[2] + xSize) <= 1.0e-8)
                        et = 6;

                    return et;
                });

                return grd;
     }
 
 }

In [8]:
public static class BoundaryValueFactory { 
    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
           stw.WriteLine("static class BoundaryValues {");
           stw.WriteLine("  static public double Zero(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");

           stw.WriteLine("  static public double pJump(double[] X) {");
           stw.WriteLine("    return 0.046 / 0.4;");
           stw.WriteLine("  }");
           
           stw.WriteLine(" static public double InitialPhi(double[] X) { ");
           stw.WriteLine("    return (((X[0] - 0.0).Pow2() + (X[1] - 0.0).Pow2() + (X[2] - 0.0).Pow2()) - 0.16);");
           //stw.WriteLine("    return -(X[0] - 0);");
           stw.WriteLine("    }"); 
           
           stw.WriteLine(" static public double InitialPhi1(double[] X) { ");
           stw.WriteLine("    return -(X[1] + 0);");
           stw.WriteLine("    }"); 

           stw.WriteLine("}"); 
           return stw.ToString();
        }
    }
   
    static public Formula Get_Zero() {
        return new Formula("BoundaryValues.Zero", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_pJump(){
        return new Formula("BoundaryValues.pJump", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialPhi(){
        return new Formula("BoundaryValues.InitialPhi", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InitialPhi1(){
        return new Formula("BoundaryValues.InitialPhi1", AdditionalPrefixCode:GetPrefixCode());
    }
}

## Create Control file

In [9]:
        public static ZLS_Control Aland( int p = 2, int AMRlvl = 0) {
            ZLS_Control C = new ZLS_Control(p);
            //C.ImmediatePlotPeriod = 1;
            //C.SuperSampling = 3;

            C.AgglomerationThreshold = 0.3;
            C.NoOfMultigridLevels = 1;

            int D = 3;

            AppControl._TimesteppingMode compMode = AppControl._TimesteppingMode.Transient;

            //_DbPath = @"\\fdyprime\userspace\smuda\cluster\cluster_db";
            //_DbPath = @"D:\local\local_Testcase_databases\Testcase_ContactLine";
            //_DbPath = @"D:\local\local_spatialConvStudy\StaticDropletOnPlateConvergence\SDoPConvDB";

            // basic database options
            // ======================
            #region db

            C.savetodb = true;
            //C.DbPath = @"\\dc1\userspace\miao\cluster\Droplet-EE-LikeEL-3D";
            C.ProjectName = "Droplet";
            C.SessionName = "Droplet-3D-6x6x3(x4)-Old-MPI16-METIS-NoG";
            //C.ProjectDescription = "Droplet running on pc";

            C.ContinueOnIoError = false;

            //C.LogValues = XNSE_Control.LoggingValues.MovingContactLine;
            //C.PostprocessingModules.Add(new MovingContactLineLogging());

            #endregion

            if(D == 3) {
                C.FieldOptions.Add("DisplacementZ", new FieldOpts() {
                    Degree = p,
                    SaveToDB = FieldOpts.SaveToDBOpt.TRUE
                });
            }

            // Physical Parameters
            // ===================
            #region physics

            double scale = 4.4175e-4; // For a droplet with radius r = 176.7 micrometre
            C.PhysicalParameters.rho_A = 10 * scale * scale * scale;
            C.PhysicalParameters.rho_B = 1260 * scale * scale * scale;
            C.PhysicalParameters.mu_A = 0.1 * scale ;
            C.PhysicalParameters.mu_B = 1.41 * scale;
            double sigma = 0.046;
            C.PhysicalParameters.Sigma = sigma;

            C.PhysicalParameters.betaS_A = 0.0;
            C.PhysicalParameters.betaS_B = 0.0;

            C.PhysicalParameters.betaL = 0.0;
            C.PhysicalParameters.theta_e = Math.PI / 2.0;

            C.PhysicalParameters.IncludeConvection = true;
            C.PhysicalParameters.Material = false; //??

            C.Material = new Solid() {
                Density = 1000 * scale * scale * scale,
                Lame2 = 1000 * scale,
                Viscosity = 1 * scale, // Real Viscosity
            };

            #endregion

            // grid generation
            // ===============
            #region grid

            C.SetGrid(GridFactory.GenerateGrid());

            #endregion

            // Initial Values
            // ==============

            C.AddInitialValue("Pressure#A", BoundaryValueFactory.Get_pJump());
            C.AddInitialValue("Pressure#B", BoundaryValueFactory.Get_Zero());
            C.AddInitialValue("Phi", BoundaryValueFactory.Get_InitialPhi());
            C.AddInitialValue("Phi2", BoundaryValueFactory.Get_InitialPhi1());
            //C.AddInitialValue(ZwoLevelSetSolver.VariableNames.SolidLevelSetCG, BoundaryValueFactory.Get_InitialPhi1());



            // boundary conditions
            // ===================
            #region BC


            C.AddBoundaryValue("wall_lower");
            C.AddBoundaryValue("pressure_outlet_upper");
            C.AddBoundaryValue("FreeSlip_left");
            C.AddBoundaryValue("FreeSlip_right");
            C.AddBoundaryValue("FreeSlip_front");
            C.AddBoundaryValue("FreeSlip_back");

            C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
            C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_Beta;
            //C.PhysicalParameters.sliplength = 0.001;

            #endregion

            // misc. solver options
            // ====================
            #region solver


            //C.AdvancedDiscretizationOptions.CellAgglomerationThreshold = 0.2;
            //C.AdvancedDiscretizationOptions.PenaltySafety = 40;
            //C.AdvancedDiscretizationOptions.UseGhostPenalties = true;

            C.NonLinearSolver.MaxSolverIterations = 160;
            C.NonLinearSolver.MinSolverIterations = 2;
            //C.Solver_MaxIterations = 50;
            C.NonLinearSolver.ConvergenceCriterion = 1e-8;
            //C.Solver_ConvergenceCriterion = 1e-8;
            C.LevelSet_ConvergenceCriterion = 1e-12;
            C.NonLinearSolver.Globalization = BoSSS.Solution.AdvancedSolvers.Newton.GlobalizationOption.Dogleg;


            //C.Option_LevelSetEvolution = (compMode == AppControl._TimesteppingMode.Steady) ? LevelSetEvolution.None : LevelSetEvolution.FastMarching;
            //C.EllipticExtVelAlgoControl.solverFactory = () => new ilPSP.LinSolvers.PARDISO.PARDISOSolver();
            //C.EllipticExtVelAlgoControl.IsotropicViscosity = 1e-3;
            //C.fullReInit = false; 

            C.AdvancedDiscretizationOptions.FilterConfiguration = CurvatureAlgorithms.FilterConfiguration.NoFilter;
            C.AdvancedDiscretizationOptions.ViscosityMode = ViscosityMode.FullySymmetric;
            C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.XNSECommon.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

            C.AdaptiveMeshRefinement = false;
            //C.activeAMRlevelIndicators.Add(new ContactPointRefiner { maxRefinementLevel = 2});
            //C.AMR_startUpSweeps = 2;
            //C.activeAMRlevelIndicators.Add(new AMRatInnerContactLine { maxRefinementLevel = AMRlvl, levelSets = new[] { 0, 1 }, FieldWidth = 1 });
            //C.AMR_startUpSweeps = AMRlvl;

            #endregion

            C.DynamicLoadBalancing_On = true;
            C.DynamicLoadBalancing_Period = 50;
            C.DynamicLoadBalancing_RedistributeAtStartup = true;

            //C.GridPartType = GridPartType.clusterHilbert;
            C.GridPartType = GridPartType.METIS;

            // Timestepping
            // ============
            #region time

            //C.CheckJumpConditions = true;

            C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
            C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
            C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
            

            C.TimesteppingMode = compMode;
            //double dt = 0.001;
            double dt = 2e-5;
            C.dtMax = dt;
            C.dtMin = dt;
            C.Endtime = 100;
            C.NoOfTimesteps = 100;
            C.saveperiod = 1;
            
            C.TracingNamespaces = "*";

            #endregion

            return C;
        }

## Send and run jobs

In [10]:
var C_CTRLFILE = Aland(2, 0);//Create control file.

Grid Edge Tags changed.


In [11]:
var JobLocal = C_CTRLFILE.CreateJob();

In [12]:
JobLocal.Status

PreActivation

In [13]:
JobLocal.NumberOfMPIProcs = 16;
JobLocal.Activate(myBatch);
JobLocal.ShowOutput();
//BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(12*60*60);

Deploying job Droplet-3D-6x6x3(x4)-Old-MPI16-METIS-NoG ... 
Opening existing database 'C:\Users\miao\Documents\Database\Droplet-EE-LikeEL-3D-Merge-Compare'.
Set Database: { Session Count = 18; Grid Count = 62; Path = \\dc3\userspace\miao\cluster\Droplet-EE-LikeEL-3D-Merge-Compare }
Grid is not in database yet...
Grid successfully saved: d2893b7f-7908-493a-97e3-683153462951


Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\miao\cluster\Droplet-EE-LikeEL-3D-Merge-Compare-ZwoLevelSetSolver2024Apr21_001618.823578
copied 46 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.

Starting external console ...
(You may close the new window at any time, the job will continue.)


In [6]:
wmg.DefaultDatabase.Sessions

#0: Droplet-EE-LikeEL-3D-Merge-Compare	Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart_Restart_Restart_Restart_Restart_Restart_Restart*	05/08/2024 19:54:15	aec7154f...
#1: Droplet-EE-LikeEL-3D-Merge-Compare	Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart_Restart_Restart_Restart_Restart_Restart*	05/06/2024 09:14:06	785e7c45...
#2: Droplet-EE-LikeEL-3D-Merge-Compare	Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart_Restart_Restart_Restart_Restart*	05/04/2024 09:59:12	cdb1f70f...
#3: Droplet-EE-LikeEL-3D-Merge-Compare	Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart_Restart_Restart_Restart*	05/02/2024 10:30:54	a606fe8c...
#4: Droplet-EE-LikeEL-3D-Merge-Compare	Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart_Restart_Restart*	04/28/2024 10:06:45	0d0fde30...
#5: Droplet-EE-LikeEL-3D-Merge-Compare	Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart_Restart*	04/24/2024 13:04:29	1a3a9cc8...
#6: Droplet-EE-LikeEL-3D-Merge-Compare	Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart*	04/21/2024 18:30:04	53e96a6d...
#7: Droplet-E

In [11]:
wmg.DefaultDatabase.Sessions[0].Timesteps.Count

6

In [12]:
var outPath = wmg.DefaultDatabase.Sessions[0].Timesteps.Every(1).Skip(4).Export().WithSupersampling(3).Do();

Starting export process... Data will be written to the directory: C:\Users\miao\AppData\Local\BoSSS\plots\sessions\Droplet-EE-LikeEL-3D-Merge-Compare__Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart_Restart_Restart_Restart_Restart_Restart_Restart__aec7154f-065e-4c3a-985e-0e2aadf6232a


## Post processing

In [ ]:
//var f = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(1).GetField("Phi");

In [ ]:
//var v = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(5).GetField("VelocityX");

## Restart

In [14]:
wmg.DefaultDatabase.Sessions

#0: Droplet-EE-LikeEL-3D-Merge-Compare	Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart_Restart_Restart_Restart_Restart_Restart_Restart*	05/08/2024 19:54:15	aec7154f...
#1: Droplet-EE-LikeEL-3D-Merge-Compare	Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart_Restart_Restart_Restart_Restart_Restart*	05/06/2024 09:14:06	785e7c45...
#2: Droplet-EE-LikeEL-3D-Merge-Compare	Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart_Restart_Restart_Restart_Restart*	05/04/2024 09:59:12	cdb1f70f...
#3: Droplet-EE-LikeEL-3D-Merge-Compare	Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart_Restart_Restart_Restart*	05/02/2024 10:30:54	a606fe8c...
#4: Droplet-EE-LikeEL-3D-Merge-Compare	Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart_Restart_Restart*	04/28/2024 10:06:45	0d0fde30...
#5: Droplet-EE-LikeEL-3D-Merge-Compare	Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart_Restart*	04/24/2024 13:04:29	1a3a9cc8...
#6: Droplet-EE-LikeEL-3D-Merge-Compare	Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart*	04/21/2024 18:30:04	53e96a6d...
#7: Droplet-E

In [15]:
var Sloc = wmg.DefaultDatabase.Sessions[0];
Sloc

Droplet-EE-LikeEL-3D-Merge-Compare	Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart_Restart_Restart_Restart_Restart_Restart_Restart*	05/08/2024 19:54:15	aec7154f...

In [16]:
var c2 = Sloc.CreateRestartControl() as ZLS_Control;

In [17]:
c2.GetType()

ZwoLevelSetSolver.ZLS_Control

In [18]:
c2.GridGuid

75453eb0-8fcc-4799-ab62-851f1d33053d

In [19]:
Sloc.Timesteps.Last().GridID

75453eb0-8fcc-4799-ab62-851f1d33053d

In [20]:
c2.GridGuid = Sloc.Timesteps.Last().GridID;

In [21]:
c2.GridGuid

75453eb0-8fcc-4799-ab62-851f1d33053d

In [22]:
//c2.AMR_startUpSweeps = 0;

In [23]:
c2.DynamicLoadBalancing_On = true;
c2.DynamicLoadBalancing_Period = 10;
c2.DynamicLoadBalancing_RedistributeAtStartup = false;
c2.GridPartType = GridPartType.METIS;

In [24]:
double dt = 1e-4;
c2.dtMax = dt;
c2.dtMin = dt;
c2.Endtime = 100;
c2.NoOfTimesteps = 100;
c2.saveperiod = 1;

In [28]:
var JobLocal2 = c2.CreateJob();
JobLocal2.Status

Error: System.ArgumentException: Job with name 'Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart_Restart_Restart_Restart_Restart_Restart_Restart_Restart' is already defined in the workflow management.
   at BoSSS.Application.BoSSSpad.Job..ctor(String name, Type solver) in C:\Users\miao\Documents\Git\BoSSS-FSI-EE\public\src\L4-application\BoSSSpad\Job.cs:line 64
   at BoSSS.Application.BoSSSpad.AppControlExtensions.CreateJob(AppControl ctrl) in C:\Users\miao\Documents\Git\BoSSS-FSI-EE\public\src\L4-application\BoSSSpad\AppControlExtensions.cs:line 62
   at Submission#30.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [27]:
JobLocal2.NumberOfMPIProcs = 8;
JobLocal2.Activate(myBatch);
JobLocal2.ShowOutput();

Deploying job Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart_Restart_Restart_Restart_Restart_Restart_Restart_Restart ... 


Error: System.IO.IOException: Database { Session Count = 26; Grid Count = 63; Path = \\dc3\userspace\miao\cluster\Droplet-EE-LikeEL-3D-Merge-Compare } is not allowed for Droplet-3D-6x6x3(x4)-Old-MPI12-METIS_Restart_Restart_Restart_Restart_Restart_Restart_Restart_Restart: Unknown (MiniBatchProcessor client @C:\Users\miao\Documents\Database\Deployment); You might either use a different database for this computation OR modify the 'AllowedDatabasesPaths' in '~/.BoSSS/etc/BatchProcessorConfig.json'.
   at BoSSS.Application.BoSSSpad.Job.FiddleControlFile(BatchProcessorClient bpc) in C:\Users\miao\Documents\Git\BoSSS-FSI-EE\public\src\L4-application\BoSSSpad\Job.cs:line 371
   at BoSSS.Application.BoSSSpad.Job.Reactivate() in C:\Users\miao\Documents\Git\BoSSS-FSI-EE\public\src\L4-application\BoSSSpad\Job.cs:line 1484
   at BoSSS.Application.BoSSSpad.Job.Activate(BatchProcessorClient bpc) in C:\Users\miao\Documents\Git\BoSSS-FSI-EE\public\src\L4-application\BoSSSpad\Job.cs:line 1446
   at 